# Bike Rental Prediction – Machine Learning Analysis

## Goals
This project analyzes a dataset of bike rentals and develops predictive models using:
- Exploratory Data Analysis (EDA)
- Ordinary Least Squares (OLS) regression on log-transformed target
- Ridge regression with hyperparameter tuning
- Cross-validation and learning curves
- A non-linear model (Random Forest or Neural Network)
- Final evaluation, discussion, limitations & future work

The objective is to create a clear, self-contained notebook demonstrating the full ML workflow.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, learning_curve
from sklearn.linear_model import PoissonRegressor, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

sns.set(style="whitegrid")

---

# 2. Data Loading and Cleaning

In this section, we load both datasets separately, inspect their structure, and perform initial preprocessing:

### Bike Counter Dataset (`train.parquet`)
- Bike rental counts at various counting stations
- Location data (latitude, longitude)
- Temporal information

### Weather Dataset (`external_data.csv`)
- Temperature, humidity, wind speed
- Precipitation, atmospheric pressure
- Weather conditions

For each dataset, we will:
- Check data types and structure
- Handle missing values
- Extract datetime features (hour, day, month, etc.)

The datasets will be merged later during **Feature Engineering** (Section 4).

## 2.1 Bike Counter Dataset

In [ ]:
# Load bike counter data
bike_data = pd.read_parquet(Path("data") / "train.parquet")
bike_data.head()

One may observe that the dataset countains two sets of bike_counts, one has an applied logarithmic scale to the bike count.
This could have been done in the past due to mutiple reasons: 
- Handle skewness: Traffic data is often characterized by hours with low to zero observations (nights, very cold, rainy weather) and hours with a high frequency (rush hour, sunnny, ...)
- reduce variance
- Mitigate the effect of events such as public holidays, sunny weather or public events on the forecasting model 

the logarithmic manipulation is applied by

$
\log(bike count + 1)
$

the + 1 is applied to make the calculation feasible for bike_counts == 0


In [ ]:
bike_data.info()

In [ ]:
bike_data.describe()

In [ ]:
print("Missing values in bike data")
print(bike_data.isna().sum())



**Observations:**
- No missing values in the bike counter dataset
- 496,827 hourly observations across 56 counters at 30 unique locations
- Target variable: `bike_count` (also available as `log_bike_count`)

In [ ]:
import holidays

# Define French holidays
FR_holidays = holidays.FR(years=range(2019, 2022))

bike_data["FR_holidays"] = bike_data["date"].dt.date.isin(FR_holidays).astype(int)
print(f"Number of rows marked as holidays: {bike_data['FR_holidays'].sum()}")

14688 data inputs are marked as holiday in the total dataset

### Extract Datetime Features

Extract temporal components from the `date` column for later analysis and modeling.

In [ ]:
def encode_dates(X):
    X = X.copy()  # Ensure we're working on a copy
    # Encode the date information
    X["year"] = X["date"].dt.year
    X["month"] = X["date"].dt.month
    X["day"] = X["date"].dt.day
    X["weekday"] = X["date"].dt.weekday  # 0=Monday, 6=Sunday
    X["hour"] = X["date"].dt.hour

    # add weekend column
    X["weekend"] = X["weekday"].isin([5, 6]).astype(int)

    return X

# apply date encoding to the bike data
bike_data = encode_dates(bike_data)

# get timesteps from data
print(f'{bike_data["date"].iloc[1] - bike_data["date"].iloc[0]}')


In [ ]:
bike_data.head()

## 2.2 Weather Dataset

Load the external weather data containing meteorological observations.


In [ ]:
weather_data = pd.read_csv(Path("data") / "external_data.csv")
weather_data.head()


In [ ]:
print(f"Weather data shape: {weather_data.shape}")
print(weather_data.columns.tolist())


In [ ]:
weather_data.info()


As the timestamp are not formatted as datetime, we bringthe date to datetime format

In [ ]:
weather_data["date"] = pd.to_datetime(weather_data["date"], errors="coerce")

In [ ]:
# Check missing values in weather data
missing_pct = (weather_data.isna().sum() / len(weather_data) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': weather_data.isna().sum(),
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print("Missing values in weather data (sorted by % missing):")
print(missing_df[missing_df['Missing Count'] > 0].head(20))


### Key Weather Features

Based on the dataset documentation, the most relevant features for bike rental prediction are:

| Column | Description | Unit |
|--------|-------------|------|
| `t` | Temperature | Kelvin |
| `u` | Humidity | % |
| `ff` | Wind speed | m/s |
| `dd` | Wind direction | degrees |
| `rr1` | Precipitation (1h) | mm |
| `pmer` | Sea-level pressure | Pa |


### Extract Datetime Features from Weather Data


In [ ]:
# apply date encoding to the weather data
weather_data = encode_dates(weather_data)

# get timestep from weather data
print(f'{weather_data["date"].iloc[1] - weather_data["date"].iloc[0]}')


the timestep of the weatherdata is 3 hours

In [ ]:
# Summary statistics for key weather features
key_weather_cols = ['t', 'u', 'ff', 'dd', 'rr1', 'pmer']
weather_data[key_weather_cols].describe()


## 2.3 Dataset Summary

Compare the date ranges to ensure temporal overlap between datasets.


In [ ]:

print("\n Bike Counter Data:")
print(f"   Rows: {len(bike_data):,}")
print(f"   Date range: {bike_data['date'].min()} to {bike_data['date'].max()}")
print(f"   Unique counters: {bike_data['counter_id'].nunique()}")
print(f"   Unique sites: {bike_data['site_id'].nunique()}")

print("\n Weather Data:")
print(f"   Rows: {len(weather_data):,}")
print(f"   Date range: {weather_data['date'].min()} to {weather_data['date'].max()}")
print(f"   Weather stations: {weather_data['numer_sta'].nunique()}")

print("\n Both datasets loaded and datetime features extracted.")
print("   Ready for EDA and Feature Engineering (merging in Section 4).")

# overlapping time range
# Print the overlapping time range between bike_data and weather_data

# Find the latest start date and earliest end date
overlap_start = max(bike_data['date'].min(), weather_data['date'].min())
overlap_end = min(bike_data['date'].max(), weather_data['date'].max())

print("\nOverlapping time range between bike data and weather data:")
print(f"   From: {overlap_start}")
print(f"   To:   {overlap_end}")



### 2.4 Merge Bike and Weather Dataset

Remove any duplicates from the the weather dataset

In [ ]:
# check for duplicates
print(weather_data.duplicated(subset="date").sum())

Bring the weather dataset to the same timestep as the bike dataset by linearly interpolating the values to match one hour timestep

In [ ]:
# Interpolate linearly to get from 3 hour data to 1 hour data
# Set date as index and resample to hourly
weather_data = weather_data.reset_index()
weather_data = weather_data.drop_duplicates(subset="date")
weather_data = weather_data.set_index('date')
weather_data = weather_data.resample('h').interpolate(method='linear')
weather_data = weather_data.reset_index()

# Check new datashape
print(f"Weather data shape after resampling: {weather_data.shape}")

Merge the data

In [ ]:
# Merge bike data with weather data using a left join
merged_data = pd.merge(bike_data, weather_data, on="date", how="left")

# Drop redundant date columns to avoid duplicates in the final dataset
merged_data = merged_data.loc[:, ~merged_data.columns.str.endswith(("_x", "_y"))]  # Change 1: Drop `_x` or `_y` suffix columns

In [ ]:
from sklearn.preprocessing import FunctionTransformer

date_encoder = FunctionTransformer(encode_dates, validate=False)
sample_encoded = date_encoder.fit_transform(merged_data[["date"]]).head()
sample_encoded

In [ ]:
# Reapply the _encode_dates function to extract date-related columns
merged_data = encode_dates(merged_data)

# Verify the new columns
print(merged_data[["date", "year", "month", "day", "weekday", "weekend"]].head())


In [ ]:
print(merged_data.shape)


---

# 3. Exploratory Data Analysis (EDA)


The goal of this section is to develop an initial understanding of the dataset by examining its structure, main variables, and early patterns that may influence model design.

---

### 3.1 Dataset Overview

We begin by loading the training dataset and inspecting its structure.  
The dataset contains hourly bicycle counts recorded at several counting stations across Paris. Each row represents the number of bicycles detected at a specific counter (`counter_id`) at a given timestamp.

In many bike-sharing and traffic forecasting problems, it is common to use a log-transformed target variable:

\[
\text{log\_bike\_count} = \ln(\text{bike\_count} + 1)
\]

This transformation is useful because:

1. **Reducing skewness:** Bike traffic has many low-count hours and a few very high-count periods.  
2. **Stabilizing variance:** Helps linear models capture relationships more effectively.  
3. **Interpretability:** Changes in log-scale correspond to percentage-level changes in demand.

Whether or not we ultimately use the log-transformation will be evaluated during the modeling phase.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sns.set_theme()

data = pd.read_parquet(Path("data") / "train.parquet")
data.head()


We now inspect the structure of the dataset:

- `data.info()` provides the data types and missing-value overview.  
- `data.nunique()` shows how many unique values exist in key fields such as counters and timestamps.

In [ ]:
data.info()
data.nunique(axis=0)


<span style="color:darkblue">

The dataset contains **56 counters** grouped into **30 sites**, meaning several counters are colocated at the same physical location.  
Coordinates and site identifiers confirm this structure.

There are **8,974 unique timestamps**, indicating broad temporal coverage.  
`bike_count` and `log_bike_count` show similar variability, each with nearly 1,000 distinct values.

</span>


#### Distribution of Activity Across Counters

Before analyzing temporal or weather-related patterns, it is useful to understand how bicycle traffic is distributed across counting stations.

Some counters record substantially higher traffic than others.  
High-volume counters typically correspond to major commuting corridors or central areas, while low-volume counters might be located in residential zones or less frequently used routes.

The following table lists the counters with the highest total recorded bike counts.

In [ ]:
(
    data.groupby("counter_id")["bike_count"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .to_frame(name="total_bike_count")
)


<span style="color:darkblue">

The busiest counters each record well over **1 million bicycles**, indicating major commuting corridors.  
Many appear in **paired IDs**, suggesting counters measuring opposite directions at the same site.

Traffic volume varies strongly across locations, reflecting substantial spatial heterogeneity in bike usage.

</span>


### 3.2 Visualizing the Data

#### 3.2.1. Spatial Distribution of Counters

We first visualize the spatial distribution of all counting stations across Paris.
Using the geographic coordinates (`latitude`, `longitude`), we place a marker for each counter on an interactive map.

This helps verify that locations are correctly recorded and provides an initial sense of where bicycle traffic is being measured in the city.

In [ ]:
import folium

# Center the map on the mean latitude and longitude of all counters
m = folium.Map(location=data[["latitude", "longitude"]].mean(axis=0), zoom_start=13)

# Add one marker per counter
for _, row in (
    data[["counter_name", "latitude", "longitude"]]
    .drop_duplicates("counter_name")
    .iterrows()
):
    folium.Marker(
        location=row[["latitude", "longitude"]].values.tolist(),
        popup=row["counter_name"],
    ).add_to(m)

m


#### 3.2.2. Temporal Patterns at a Single Counter

To gain a more detailed understanding of temporal dynamics, we focus on one representative counting station.
By inspecting its time series over the full observation period, we can visually assess long-term trends, seasonality, and noise in the bike counts.


In [ ]:
# Select a specific counter (example: a busy central counter)
mask = data["counter_name"] == "Totem 73 boulevard de Sébastopol S-N"

# Filter and aggregate (here aggregation is trivial if each timestamp appears once)
data_mask = data[mask].copy()
data_mask["date"] = pd.to_datetime(data_mask["date"])  # Ensure datetime type
data_agg = data_mask.groupby("date", as_index=False)["bike_count"].sum()

# Plot bike counts over time for this counter
data_agg.plot(x="date", y="bike_count", title="Bike Count Over Time", legend=True)


We next aggregate the time series at a weekly level for the same counter.
This smooths out hourly fluctuations and highlights broader patterns such as seasonal trends or sustained growth/decline in bike usage.

In [ ]:
mask = data["counter_name"] == "Totem 73 boulevard de Sébastopol S-N"

(
    data[mask]
    .groupby(pd.Grouper(freq="1w", key="date"))[["bike_count"]]
    .sum()
    .plot(title="Weekly Aggregated Bike Count")
)


<span style="color:darkblue">

Weekly totals show a clear **seasonal pattern**: bike usage drops sharply in winter months (December–February) and rises again approaching late spring and summer.  
This reflects typical weather-driven cycling behaviour, with warmer periods supporting higher mobility across the city.

</span>

#### 3.2.3. Zooming into a Single Week

To better understand intra-week patterns, we zoom into one specific week for the same counter.
This allows us to compare workdays and weekends in terms of hourly or daily traffic.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Filter for the specific counter and a chosen date range
mask = (
    (data["counter_name"] == "Totem 73 boulevard de Sébastopol S-N")
    & (data["date"] > pd.to_datetime("2021-03-01"))
    & (data["date"] < pd.to_datetime("2021-03-08"))
)

data_filtered = data[mask].copy()

# Optionally aggregate (depending on time resolution)
data_filtered = data_filtered.groupby("date", as_index=False)["bike_count"].sum()

# Plot the filtered data
data_filtered.plot(x="date", y="bike_count", ax=ax, marker=".", legend=False)
ax.set_title("Bike Count from March 1 to March 8, 2021")
ax.set_ylabel("Bike Count")
ax.set_xlabel("Date")
plt.show()


<span style="color:darkblue">

Within a single week, strong **weekday rush-hour cycles** appear with two clear peaks per day (morning and evening commutes). On weekends, like on 6th and 7th March, the shape of the during the day looks different. In terms of daily peaks, there seem to be slight differences when comparing Monday-Wednesday with Thursday & Friday.

</span>

#### 3.2.4. Calendar Feature Encoding

Bike usage is strongly driven by time-related effects such as hour of the day, day of the week, and season.
To capture these effects, we decompose the `date` variable into several calendar components:

- `year`
- `month`
- `day`
- `weekday` (0 = Monday, ..., 6 = Sunday)
- `hour`

These features will later be used as predictors in our regression models.


In [ ]:
def _encode_dates(X: pd.DataFrame) -> pd.DataFrame:
    """Add calendar features derived from the 'date' column."""
    X = X.copy()
    X["date"] = pd.to_datetime(X["date"])
    X["year"] = X["date"].dt.year
    X["month"] = X["date"].dt.month
    X["day"] = X["date"].dt.day
    X["weekday"] = X["date"].dt.weekday  # 0=Monday, 6=Sunday
    X["hour"] = X["date"].dt.hour
    return X

data = _encode_dates(data)


#### 3.2.5. Aggregate Weekday Patterns

We now aggregate the total bike counts by weekday, summing over all counters and all timestamps.
This provides an overall view of how bicycle usage varies from Monday to Sunday across the entire network.


In [ ]:
# Aggregate bike counts by weekday (0 = Monday, ..., 6 = Sunday)
weekday_aggregates = data.groupby("weekday")["bike_count"].sum()

# Define order and labels
weekday_order = [0, 1, 2, 3, 4, 5, 6]
weekday_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

weekday_aggregates = weekday_aggregates.reindex(weekday_order)

plt.figure(figsize=(10, 6))
weekday_aggregates.index = weekday_names
weekday_aggregates.plot(kind="bar", edgecolor="black")
plt.title("Total Bike Count by Weekday")
plt.ylabel("Total Bike Count")
plt.xlabel("Day of the Week")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


<span style="color:darkblue">

Aggregated across all counters, **Tuesday to Thursday** show the highest bike usage, while Monday and Friday are slightly lower. Weekend days exhibit a pronounced reduction in total counts, consistent with fewer commuting trips and more variable recreational activity.

</span>

#### 3.2.6. Weekday Patterns by Counter (Normalized to Monday)

To examine whether weekday patterns are consistent across locations, we compute the total bike count per weekday for each counter and normalize these values by the respective Monday count.

This yields a percentage index where Monday corresponds to 100%, and other days indicate relative increases or decreases compared to Monday for each station.


In [ ]:
# Sum bike counts by station and weekday
station_weekday_counts = data.groupby(["counter_name", "weekday"])["bike_count"].sum().unstack()

# Normalize by Monday (weekday = 0) for each station
normalized_counts = station_weekday_counts.div(station_weekday_counts[0], axis=0) * 100

# Plot normalized weekday distributions for all stations
plt.figure(figsize=(12, 6))
for station in normalized_counts.index:
    plt.plot(normalized_counts.columns, normalized_counts.loc[station], alpha=0.6)

plt.title("Normalized Bike Counts by Weekday (Percentage of Monday) for Each Counter")
plt.ylabel("Percentage of Monday's Count (%)")
plt.xlabel("Weekday (0=Monday, ..., 6=Sunday)")
plt.tight_layout()
plt.show()


<span style="color:darkblue">

Across stations, weekday profiles are remarkably consistent: counts remain close to **110–120% of Monday levels** from Tuesday to Thursday. In contrast, Saturday and Sunday display far greater variation between counters, suggesting location-specific differences in recreational cycling behaviour.

Maybe it would makes sense to group Tuesday until Thursday as one category, given they are all at an equal level in terms of counts.

</span>

#### 3.2.7. Distribution of the Target Variable

Understanding the distribution of the target variable is important before fitting models.
Least-squares–based regression methods (such as OLS or Ridge) assume errors that are approximately normally distributed, which is often easier to satisfy when the target itself is closer to a symmetric distribution.

We therefore begin by inspecting the raw distribution of `bike_count`.


In [ ]:
ax = sns.histplot(data, x="bike_count", kde=True, bins=50)
plt.title("Distribution of Raw Bike Counts")
plt.show()


<span style="color:darkblue">

The distribution of `bike_count` is extremely right-skewed: most observations correspond to very low bicycle flows, while a small number of hours show exceptionally high usage. This long-tailed behaviour is typical of mobility datasets and suggests that a direct modeling of `bike_count` may violate normal-error assumptions.

</span>

Because of the strong skewness, applying a log-transformation can help stabilize variance  
and make the distribution more symmetric. We now examine the distribution of the transformed  
variable `log_bike_count`, defined as:

$$
\text{log\_bike\_count} = \ln(\text{bike\_count} + 1)
$$


In [ ]:
ax = sns.histplot(data, x="log_bike_count", kde=True, bins=50)
plt.title("Distribution of Log-Transformed Bike Count")
plt.show()

<span style="color:darkblue">

The log-transformation substantially reduces skewness, producing a more balanced and bell-shaped distribution.  
Although still not perfectly Gaussian, the transformed variable aligns more closely with the assumptions of linear models and is therefore a promising candidate for the modeling phase.

</span>

### 3.3 Correlation Analysis

To better understand the relationships between the target variable and the time-related features, we compute pairwise Pearson correlations between:

- `log_bike_count` (transformed target),
- the original `bike_count`,
- and the calendar features `hour`, `weekday`, `month`, and `year`.

Visualising these correlations in a heatmap helps identify which variables appear most informative for explaining variation in bicycle usage.


In [ ]:
# Select relevant features for correlation analysis
correlation_features = ["log_bike_count", "hour", "weekday", "month", "year", "bike_count"]

# Compute the correlation matrix
correlation_matrix = data[correlation_features].corr()

# Plot the correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Heatmap with Log Bike Count")
plt.show()


<span style="color:darkblue">

`log_bike_count` shows its strongest association with the **original bike count**, as expected, and a **moderate positive correlation with the hour of the day**, reflecting the strong daily cycle in bike usage.  
Correlations with `weekday`, `month`, and `year` are comparatively weak at this aggregated level, indicating that **short-term temporal structure (hourly effects)** dominates over coarser calendar effects.

</span>

### 3.4 Time-of-Day Phases

The correlation analysis suggests that the hour of the day is an important driver of bike usage.  
As an exploratory step, we group hours into broader **time-of-day phases** (Night, Morning, Midday, Afternoon, Evening, Late Evening) and study how the average `log_bike_count` behaves across these categories.

This can be seen as a simple form of feature engineering that replaces the raw hour with a coarser, interpretable time-of-day variable.

In [ ]:
def _add_time_phases(X: pd.DataFrame) -> pd.DataFrame:
    """Add a categorical 'time_of_day' feature based on the hour of day."""
    X = X.copy()
    # Create the 'time_of_day' categorical variable
    X["time_of_day"] = pd.cut(
        X["hour"],
        bins=[0, 5, 9, 13, 17, 20, 23],
        labels=["Night", "Morning", "Midday", "Afternoon", "Evening", "Late Evening"],
        right=True,
    )
    return X

# Apply the function to the dataset
data = _add_time_phases(data)

# Quick preview
data[["hour", "time_of_day"]].head()


We now compare the mean `log_bike_count` computed:

- for each individual hour of the day, and  
- for each time-of-day phase.

This allows us to assess whether aggregating hours into broad phases preserves or hides important hourly structure.

In [ ]:
# Mean and median by time-of-day phase
time_of_day_stats = data.groupby("time_of_day")["log_bike_count"].agg(["mean", "median"]).reset_index()

# Mean by hour of day
hourly_stats = data.groupby("hour")["log_bike_count"].mean().reset_index()

# Two subplots: hourly mean and phase mean
fig, ax = plt.subplots(2, 1, figsize=(8, 8), sharey=True)

# Plot hourly mean
ax[0].plot(hourly_stats["hour"], hourly_stats["log_bike_count"], label="Hourly Mean", marker="o")
ax[0].set_title("Hourly Mean of Log Bike Count")
ax[0].set_ylabel("Log Bike Count")
ax[0].set_xlabel("Hour of the Day")
ax[0].grid(True)

# Plot time-of-day mean as bar chart
time_of_day_stats.plot(
    x="time_of_day",
    y="mean",
    kind="bar",
    ax=ax[1],
    color="red",
    legend=False,
    edgecolor="black",
)
ax[1].set_title("Time of Day Mean of Log Bike Count")
ax[1].set_ylabel("Log Bike Count")
ax[1].set_xlabel("Time of Day")
ax[1].set_xticks(range(len(time_of_day_stats["time_of_day"])))
ax[1].set_xticklabels(time_of_day_stats["time_of_day"], rotation=45)

plt.tight_layout()
plt.show()


<span style="color:darkblue">

The hourly mean plot reveals a detailed **double-peak structure** with sharp morning and evening rush hours, and lower activity at night.  
When hours are aggregated into coarse time-of-day phases, these nuances largely disappear: phase means smooth over intra-phase variability and fail to fully capture the commuting peaks.

This suggests that, for modeling purposes, retaining the **original hourly resolution** (or using more refined encodings such as cyclical transforms) is preferable to replacing it with broad time-of-day categories.

</span>

---

# 4. Feature Engineering

To improve predictive performance, we engineer additional features:

### Time-based features
- Hour of day  
- Weekday vs weekend  
- Month / season  

### Weather transformations
- Non-linear terms (e.g., temperature²)  
- Binary indicators (e.g., extreme weather)

### Optional future extensions
- Lag features (previous hour/day rentals)  
- Rolling averages  

These engineered features form the input for the machine learning models.

In [ ]:
## 4.1 Check merged data and handle missing values



In [ ]:
# Check for missing values after merge
missing_after_merge = merged_data.isna().sum()
missing_cols = missing_after_merge[missing_after_merge > 0]
print(f"Columns with missing values after merge: {len(missing_cols)}")
print(missing_cols.sort_values(ascending=False).head(15))


## 4.2 Temperature Conversion and Weather Transformations

Convert temperature from Kelvin to Celsius for better interpretability and create derived weather features.


In [ ]:
# Convert temperature from Kelvin to Celsius
merged_data['temp_celsius'] = merged_data['t'] - 273.15

# Create temperature squared (captures non-linear relationship)
merged_data['temp_squared'] = merged_data['temp_celsius'] ** 2

## 4.3 Cyclical Encoding for Time Features

Encode cyclical features (hour, weekday, month) using sine and cosine transformations. This preserves the cyclical nature of time (e.g., hour 23 is close to hour 0).

In [ ]:
# Cyclical encoding for hour (24-hour cycle)
merged_data['hour_sin'] = np.sin(2 * np.pi * merged_data['hour'] / 24)
merged_data['hour_cos'] = np.cos(2 * np.pi * merged_data['hour'] / 24)

# Cyclical encoding for weekday (7-day cycle)
merged_data['weekday_sin'] = np.sin(2 * np.pi * merged_data['weekday'] / 7)
merged_data['weekday_cos'] = np.cos(2 * np.pi * merged_data['weekday'] / 7)

# Cyclical encoding for month (12-month cycle)
merged_data['month_sin'] = np.sin(2 * np.pi * merged_data['month'] / 12)
merged_data['month_cos'] = np.cos(2 * np.pi * merged_data['month'] / 12)

print("Cyclical features created:")
merged_data[['hour', 'hour_sin', 'hour_cos', 'weekday', 'weekday_sin', 'weekday_cos']].head(10)


## 4.4 Season Feature

Create a season feature based on month.


In [ ]:
# Create season feature (meteorological seasons)
def get_season(month):
    if month in [12, 1, 2]:
        return 0  # Winter
    elif month in [3, 4, 5]:
        return 1  # Spring
    elif month in [6, 7, 8]:
        return 2  # Summer
    else:
        return 3  # Autumn

merged_data['season'] = merged_data['month'].apply(get_season)

# Cyclical encoding for season (4-season cycle)
merged_data['season_sin'] = np.sin(2 * np.pi * merged_data['season'] / 4)
merged_data['season_cos'] = np.cos(2 * np.pi * merged_data['season'] / 4)

# Season distribution
print("Season distribution:")
print(merged_data['season'].value_counts().sort_index())
print("\n0=Winter, 1=Spring, 2=Summer, 3=Autumn")

# Show cyclical encoding
print("\nSeason cyclical encoding:")
print(merged_data[['season', 'season_sin', 'season_cos']].drop_duplicates().sort_values('season'))


## 4.5 Feature Selection and Final Dataset

Select the relevant features for modeling and prepare the feature matrix `X` and target `y`.


In [ ]:
# Define feature columns for modeling
feature_cols = [
    # Time features (cyclical)
    'hour_sin', 'hour_cos',
    'weekday_sin', 'weekday_cos', 
    'month_sin', 'month_cos',
    'season_sin', 'season_cos',
    # Time features (binary)
    'weekend', 'FR_holidays',
    # Weather features
    'temp_celsius', 'temp_squared',
    'u',  # humidity
    'ff',  # wind speed
    'rr1',  # precipitation
    'pmer',  # pressure
    # Location
    'latitude', 'longitude'
]

# Target variable
target_col = 'bike_count'

print(f"Number of features: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")


In [ ]:
# Check for missing values in selected features
missing_in_features = merged_data[feature_cols].isna().sum()
print("Missing values in feature columns:")
print(missing_in_features[missing_in_features > 0])

# Fill missing weather values with median (if any)
for col in feature_cols:
    if merged_data[col].isna().sum() > 0:
        merged_data[col] = merged_data[col].fillna(merged_data[col].median())
        print(f"Filled {col} with median")

print(f"\nTotal missing after filling: {merged_data[feature_cols].isna().sum().sum()}")


## 2.X Scale the data

In [ ]:


# Initialize the scaler
scaler = StandardScaler()

# Fit scaler on training data, but here we scale all for simplicity before split
merged_data[feature_cols] = scaler.fit_transform(merged_data[feature_cols])

print("Feature columns scaled using StandardScaler.")


In [ ]:
X = merged_data[feature_cols]
y = merged_data[target_col]


In [ ]:
# create train and validaiton set
cutoff_date = merged_data['date'].max() - pd.Timedelta("30 days")
mask = merged_data['date'] < cutoff_date
X_train, y_train = merged_data[mask][feature_cols], merged_data[mask][target_col]
X_val, y_val = merged_data[~mask][feature_cols], merged_data[~mask][target_col]

---

# 5. Baseline Model — Poisson Least Squares (OLS)

We train a simple Poisson Regression model using:

- Log-transformed rental counts (`log1p(count)`)
- Feature set engineered in Section 4  
- Evaluation on test data using RMSE and R²  

This acts as the baseline for comparing more advanced models.

### Interpretation
We analyze regression coefficients to understand which features increase or decrease expected rental demand.

In [ ]:
model = PoissonRegressor()
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

model.score(X_val, y_val)

In [ ]:
# To color points by their value, use the `c` parameter and a colormap.
# For example, color by the true value (y_val) using a colormap like 'viridis':
plt.scatter(y_val, y_pred, c=y_val, cmap='viridis', alpha=0.7)
# If you want to color by predicted values instead, use c=y_pred.
plt.xlabel("Acutal bike counts")
plt.ylabel("predicted bike counts")
plt.title("Actual vs Predicted")
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', label='Perfect Prediction')
plt.legend()

plt.show()


In [ ]:
# Time series visualization: Actual vs Predicted for a sample counter
val_data = merged_data[~mask].copy()
val_data['y_pred'] = y_pred

# Pick one counter for visualization
sample_counter = val_data['counter_id'].iloc[0]
sample_data = val_data[val_data['counter_id'] == sample_counter].sort_values('date')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(sample_data['date'], sample_data['log_bike_count'], label='Actual', alpha=0.7)
ax.plot(sample_data['date'], sample_data['y_pred'], label='Predicted', alpha=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('log(bike_count)')
ax.set_title(f'Actual vs Predicted over Time (Counter: {sample_counter})')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


---

# 6. Regularized Model — Ridge Regression

Ridge Regression extends linear regression by adding L2 regularization, which penalizes large coefficients and reduces overfitting.  
Because our feature space contains many one-hot encoded variables (dates, counters, categories), regularization helps stabilize the model and improve generalization.

In this section we:

1. Prepare the data using a temporal train/validation split.  
2. Build a preprocessing pipeline including one-hot encoding and feature scaling.  
3. Train a Ridge regression model on the log-transformed target.  
4. Evaluate its performance and compare it with a simple baseline.  
5. Visualize predictions for a sample counter.  


### 6.1 Train–Validation Split

We use a time-based split so the model is always trained on past data and validated on future data.  
This respects the chronological structure of bike traffic and avoids data leakage.


In [ ]:
X, y = get_train_data()
X_train, y_train, X_valid, y_valid = train_test_split_temporal(X, y)

print(
    f'Train: n_samples={X_train.shape[0]},  {X_train["date"].min()} to {X_train["date"].max()}'
)
print(
    f'Valid: n_samples={X_valid.shape[0]},  {X_valid["date"].min()} to {X_valid["date"].max()}'
)


<span style="color:darkblue">

The training set covers almost one full year of data, while the validation period corresponds to August–September 2021.  
This ensures a realistic forecasting setup: the model learns from past months to predict future traffic.

</span>

### 6.2 Ridge Regression Pipeline

We construct a pipeline that:

- expands the date column into multiple temporal features,  
- one-hot encodes categorical identifiers,  
- scales numerical weather variables,  
- and finally fits a Ridge regression model.

This ensures that preprocessing is applied consistently during training, validation, and later during cross-validation.

In [ ]:
pipe = make_pipeline(date_encoder, preprocessor, Ridge())
pipe.fit(X_train, y_train)

<span style="color:darkblue">

The pipeline successfully fits the Ridge model on more than 450,000 training samples using all engineered features.

</span>

### 6.3 Performance Evaluation

We compute RMSE on the training and validation sets to measure predictive accuracy.  
The target is log-transformed (`log_bike_count`), so RMSE values should be interpreted in log-space.

In [ ]:
print(
    f"Train set, RMSE={mean_squared_error(y_train, pipe.predict(X_train), squared=False):.2f}"
)
print(
    f"Valid set, RMSE={mean_squared_error(y_valid, pipe.predict(X_valid), squared=False):.2f}"
)

<span style="color:darkblue">

The validation RMSE (≈0.73) is noticeably better than the baseline mean-predictor RMSE (≈1.44), showing that Ridge captures important temporal and weather-driven structure.  
The small train–valid gap indicates limited overfitting thanks to the L2 penalty.

</span>

### 6.4 Comparison with Baseline Mean Predictor

We compare Ridge to a naive baseline that always predicts the average log-bike-count from the training set.

In [ ]:
print("Baseline mean prediction.")
print(
    f"Train set, RMSE={mean_squared_error(y_train, np.full(y_train.shape, y_train.mean()), squared=False):.2f}"
)
print(
    f"Test set, RMSE={mean_squared_error(y_valid, np.full(y_valid.shape, y_valid.mean()), squared=False):.2f}"
)

<span style="color:darkblue">

Ridge reduces the RMSE on the validation set from approximately **1.44** (baseline mean predictor) to about **0.74**, representing a reduction of nearly **50%**.  
This demonstrates that the model captures meaningful temporal and weather-related structure, whereas the baseline simply reproduces the global average.

</span>

### 6.5 Visual Inspection of Predictions

We visualize predictions on a single counter for a one-week period.  
Predictions are exponentiated back to the original scale (`bike_count = exp(log_bike_count) - 1`).

In [ ]:
# Filter validation data for one counter and a specific week
mask = (
    (X_valid["counter_name"] == "Totem 73 boulevard de Sébastopol S-N")
    & (X_valid["date"] > pd.to_datetime("2021/09/01"))
    & (X_valid["date"] < pd.to_datetime("2021/09/08"))
)

# Build a Series for y_valid with the same index as X_valid
y_valid_series = pd.Series(y_valid, index=X_valid.index, name="log_bike_count")

# Create df_viz only on the masked subset
df_viz = X_valid.loc[mask].copy()
df_viz["log_bike_count"] = y_valid_series.loc[mask]

# Back-transform from log to original scale
df_viz["bike_count"] = np.exp(df_viz["log_bike_count"]) - 1
df_viz["bike_count (predicted)"] = np.exp(pipe.predict(df_viz)) - 1

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
df_viz.plot(x="date", y="bike_count", ax=ax)
df_viz.plot(x="date", y="bike_count (predicted)", ax=ax, ls="--")
ax.set_title("Predictions with Ridge")
ax.set_ylabel("bike_count")
plt.show()


<span style="color:darkblue">

Ridge captures the daily traffic pattern well, especially the morning and evening peaks.  
However, peak magnitudes tend to be underestimated — a typical limitation of linear models that cannot model sharp nonlinear spikes.

</span>

### 6.6 Time-Series Cross-Validation

We evaluate the model using a rolling time-series split to measure robustness across the entire year.

In [ ]:
scores = cross_val_score(
    pipe, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error"
)
print("RMSE: ", scores)
print(f"RMSE (all folds): {-scores.mean():.3} ± {(-scores).std():.3}")

<span style="color:darkblue">

Cross-validated RMSE values have an average of **0.92 ± 0.08** across the six folds.  
The variability across folds reflects seasonal changes: winter months typically exhibit higher unpredictability due to greater weather-driven fluctuations, while summer traffic patterns are more regular.

</span>

### 6.7 Hyperparameter Tuning for Ridge (Alpha)

We test multiple values of the regularization parameter α using GridSearchCV.

In [ ]:
print("Best alpha:", grid.best_params_['ridge__alpha'])
print("Best RMSE:", -grid.best_score_)

<span style="color:darkblue">

All tested α values (0.01–100) produce nearly identical RMSE scores (~0.898), indicating that the model is largely insensitive to the regularization strength.This suggests that most predictive power comes from the feature encoding itself rather than from tuning the Ridge penalty.

</span>

---

# 7. Decision Tree Regression

Decision Trees model non-linear relationships by recursively splitting the feature space. They can capture patterns that linear models miss (e.g., different rush-hour profiles), but they are also prone to **overfitting**, especially with deep trees.

In this section, we:

1. Build a Decision Tree model using the full feature set.  
2. Tune the key hyperparameter `max_depth`.  
3. Validate performance with time-series cross-validation.  
4. Visualize predictions for one example counter.



### 7.1 Preprocessing and Model Setup

We reuse the same preprocessing strategy as in Ridge Regression:

- One-hot encode date and categorical variables  
- Standardize numerical weather variables  
- Keep binary indicators as-is  
- Encode date components via `_encode_dates()`

We begin by fitting a Decision Tree with a moderate depth (`max_depth=15`).

In [ ]:
date_encoder = FunctionTransformer(_encode_dates)
date_cols = _encode_dates(X_train[["date"]]).columns.tolist()
date_cols = [c for c in date_cols if c != "day"]  # Exclude 'day' to reduce sparsity

categorical_cols = ["counter_name", "site_name"]
categorical_encoder = OneHotEncoder(handle_unknown="ignore")

numerical_corr_cols = ["u","t","tx12","tn12","rafper",
                       "td","raf10","ff","nnuage3","vv"]  # Weather features

numerical_encoder = make_pipeline(
    SimpleImputer(strategy="mean"),
    StandardScaler()
)

binary_cols = ["weekend", "FR_holidays"]

preprocessor = ColumnTransformer([
    ("date", OneHotEncoder(handle_unknown="ignore"), date_cols),
    ("cat", categorical_encoder, categorical_cols),
    ("num", numerical_encoder, numerical_corr_cols),
    ("binary", "passthrough", binary_cols)
])

regressor = DecisionTreeRegressor(max_depth=15, random_state=42)
pipe = make_pipeline(date_encoder, preprocessor, regressor)

pipe.fit(X_train, y_train)

print(f"Train RMSE = {mean_squared_error(y_train, pipe.predict(X_train), squared=False):.2f}")
print(f"Valid RMSE = {mean_squared_error(y_valid, pipe.predict(X_valid), squared=False):.2f}")


<span style="color:darkblue">

The Decision Tree with max_depth=15 achieves **Train RMSE ≈ 0.83** and **Validation RMSE ≈ 0.80**, already improving over Ridge (0.74). The gap between train and validation RMSE indicates beginning overfitting but still acceptable.

</span>

### 7.2 Hyperparameter Tuning: Max Depth

`max_depth` controls how complex the tree is:

- **Small depth** → underfitting  
- **Large depth** → overfitting  
- **Too large depth** → perfect fit on training set but poor generalization

We evaluate RMSE for depths from 1 to 46.

In [ ]:
# 1. Preprocess training data ONCE
preprocessor = ColumnTransformer([
    ("date", OneHotEncoder(handle_unknown="ignore"), date_cols),
    ("cat", categorical_encoder, categorical_cols),
    ("num", numerical_encoder, numerical_corr_cols),
    ("binary", "passthrough", binary_cols)
])

# Apply date encoding separately
X_train_enc = date_encoder.fit_transform(X_train)
X_train_pre = preprocessor.fit_transform(X_train_enc)

X_valid_enc = date_encoder.transform(X_valid)
X_valid_pre = preprocessor.transform(X_valid_enc)

print("Preprocessed shapes:", X_train_pre.shape, X_valid_pre.shape)

# 2. Tune depth MUCH faster
max_depth_values = range(1, 51, 5)
train_rmse = []
valid_rmse = []

for depth in max_depth_values:
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train_pre, y_train)

    train_rmse.append(
        mean_squared_error(y_train, model.predict(X_train_pre), squared=False)
    )

    valid_rmse.append(
        mean_squared_error(y_valid, model.predict(X_valid_pre), squared=False)
    )


<span style="color:darkblue">

Validation RMSE decreases steadily up to around **depth ≈ 36**, after which it plateaus. Training RMSE continues to decrease sharply, highlighting strong overfitting. Depth=36 gives the best trade-off and will be used for cross-validation.

</span>

### 7.3 Cross-Validation (max_depth = 36)

We now validate the chosen max_depth using TimeSeriesSplit.  
This helps assess how stable the model is across the entire year.


In [ ]:
best_depth = 36
regressor = DecisionTreeRegressor(max_depth=best_depth, random_state=42)

pipe = make_pipeline(date_encoder, preprocessor, regressor)

cv = TimeSeriesSplit(n_splits=6)
scores = cross_val_score(pipe, X_train, y_train,
                         cv=cv,
                         scoring="neg_root_mean_squared_error")

print("RMSE per fold:", -scores)
print(f"Mean RMSE = {-scores.mean():.3f} ± {(-scores).std():.3f}")


<span style="color:darkblue">

Cross-validated RMSE averages **≈ 0.90**, similar to Ridge. Although tuning improves fit, the Decision Tree still suffers from high variance across folds, meaning performance is unstable across different periods of the year.

</span>


### 7.4 Example Week: Actual vs Predicted

To understand model behavior, we visualize predictions for the counter **“Totem 73 boulevard de Sébastopol S-N”** over one week.

In [ ]:
# Refit the model with best depth
regressor = DecisionTreeRegressor(max_depth=36, random_state=42)
pipe = make_pipeline(date_encoder, preprocessor, regressor)
pipe.fit(X_train, y_train)

# Filter visualization period
mask = (
    (X_valid["counter_name"] == "Totem 73 boulevard de Sébastopol S-N") &
    (X_valid["date"] > "2021-09-01") &
    (X_valid["date"] < "2021-09-08")
)

df_viz = X_valid.loc[mask].copy()
df_viz["bike_count"] = np.exp(y_valid[mask]) - 1
df_viz["bike_count_predicted"] = np.exp(pipe.predict(X_valid[mask])) - 1

plt.figure(figsize=(12,4))
plt.plot(df_viz["date"], df_viz["bike_count"], label="Actual")
plt.plot(df_viz["date"], df_viz["bike_count_predicted"], "--", label="Predicted")
plt.title("Decision Tree Predictions for Example Week")
plt.ylabel("bike_count")
plt.legend()
plt.show()


<span style="color:darkblue">

The model captures the general weekday cycling rhythm well but **misses evening peaks** and shows weaker performance on weekends. This is typical for Decision Trees trained on sparse one-hot encoded data: they cannot smooth transitions and struggle with unusual patterns.

</span>


---

# 8. Random Forest

Using max_depth = 36 from single decision trees.

### Initial Trial: n_estimators = 3

In [ ]:
from sklearn.ensemble import RandomForestRegressor


preprocessor = ColumnTransformer(
    [
        ("date", OneHotEncoder(handle_unknown="ignore"), date_cols),
        ("cat", categorical_encoder, categorical_cols),
        # ("num", numerical_encoder, numerical_corr_cols), 
        ("binary", "passthrough", binary_cols)
    ]
)

regressor = RandomForestRegressor(random_state=42, max_depth=36, n_estimators=3, n_jobs=-1)


start = time()
pipe = make_pipeline(date_encoder, preprocessor, regressor)
pipe.fit(X_train, y_train)
elapsed_time = time() - start

print(f"Training time for random forest: {elapsed_time:.2f} seconds")

print(f"Train set, RMSE={mean_squared_error(y_train, pipe.predict(X_train), squared=False):.2f}")
print(f"Valid set, RMSE={mean_squared_error(y_valid, pipe.predict(X_valid), squared=False):.2f}")

### Cross-Validation: n_estimators = 7

In [ ]:
cv = TimeSeriesSplit(n_splits=6)

# When using a scorer in scikit-learn it always needs to be better when smaller, hence the minus sign.
scores = cross_val_score(
    pipe, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
)
print("RMSE: ", scores)
print(f"RMSE (all folds): {-scores.mean():.3} ± {(-scores).std():.3}")

**Observation:** Only slight improvement over Ridge and Decision Tree. CV RMSE range: 0.7–1.0.

### Final Model: Random Forest with n_estimators = 7

In [ ]:
date_encoder = FunctionTransformer(_encode_dates)
date_cols = _encode_dates(X_train[["date"]]).columns.tolist()
date_cols = [col for col in date_cols if col != "day"] # excude day in one hot encoding

categorical_encoder = OneHotEncoder(handle_unknown="ignore")
categorical_cols = ["counter_name", "site_name"]

numerical_encoder = make_pipeline(
    SimpleImputer(strategy="mean"),  # Replace NaNs of weather data with the mean 
    StandardScaler())
numerical_corr_cols = ["u", "t", "tx12", "tn12", "rafper", "td", "raf10", "ff", "nnuage3", "vv"] # Weather columns with correlation >|0.1| (see EDA)

binary_cols = ["weekend", "FR_holidays"] # No transformation required, they are already binary


preprocessor = ColumnTransformer(
    [
        ("date", OneHotEncoder(handle_unknown="ignore"), date_cols),
        ("cat", categorical_encoder, categorical_cols),
        # ("num", numerical_encoder, numerical_corr_cols), 
        ("binary", "passthrough", binary_cols)
    ]
)

regressor = RandomForestRegressor(random_state=42, max_depth=36, n_estimators=7, n_jobs=-1)


start = time()
pipe = make_pipeline(date_encoder, preprocessor, regressor)
pipe.fit(X_train, y_train)
elapsed_time = time() - start

print(f"Training time for random forest: {elapsed_time:.2f} seconds")

print(f"Train set, RMSE={mean_squared_error(y_train, pipe.predict(X_train), squared=False):.2f}")
print(f"Valid set, RMSE={mean_squared_error(y_valid, pipe.predict(X_valid), squared=False):.2f}")

**Note:** Minimal improvement from 3 to 7 estimators, but increased robustness is expected.

In [ ]:
fig, ax = plt.subplots()

df_viz = pd.DataFrame({"y_true": y_valid, "y_pred": pipe.predict(X_valid)}).sample(
    10000, random_state=0
)

df_viz.plot.scatter(x="y_true", y="y_pred", s=8, alpha=0.1, ax=ax)

### Save the train estimators for the final script

In [ ]:
from joblib import dump

# Save the trained Random Forest model to a file
model_filename = "model1.joblib"
dump(pipe, model_filename)

print(f"Trained model saved to {model_filename}")

# 9. Apply to test data 

Not really necessary as it is included in a separate .py file. This is just to enable easy .csv file creation.

In [ ]:
test_data = pd.read_parquet(Path("data") / "final_test.parquet")

#### Same feature manipulation as for train data (without weather data as not used in final model)

In [ ]:
def _encode_dates(X):
    X = X.copy()  # Ensure we're working on a copy
    # Encode the date information
    X["year"] = X["date"].dt.year
    X["month"] = X["date"].dt.month
    X["day"] = X["date"].dt.day
    X["weekday"] = X["date"].dt.weekday  # 0=Monday, 6=Sunday
    X["hour"] = X["date"].dt.hour
    # Keep the rest of the columns as they are
    return X

# Apply the encoding function to the dataset
test_data = test_data.copy()  # Ensure we're working on a copy
test_data = _encode_dates(test_data)

test_data["weekend"] = (test_data["weekday"] > 4).astype(int)  # 1 stands for weekend, 0 stands for no weekend
test_data.head()

In [ ]:
print(f"minimum date: {test_data["date"].min()}")
print(f"maximum date: {test_data["date"].max()}")

In [ ]:
import holidays

# Define French holidays
FR_holidays = holidays.FR(years=range(2019, 2022))

test_data["FR_holidays"] = test_data["date"].dt.date.isin(FR_holidays).astype(int)
print(f"Number of rows marked as holidays: {test_data['FR_holidays'].sum()}")

#### Run the final model

In [ ]:
X_test = test_data

In [ ]:
y_pred = pipe.predict(X_test)
results = pd.DataFrame(
    dict(
        Id=np.arange(y_pred.shape[0]),
        log_bike_count=y_pred,
    )
)
results.to_csv("submission1_Forest,d=36,n=7 .csv", index=False)

---

# 7. Model Evaluation: Cross-Validation and Learning Curves

We evaluate the robustness and generalization of the models using:

### Cross-validation
- Compute cross-validated RMSE  
- Compare OLS and Ridge  

### Learning curves
Plot training and validation error as a function of training data size, allowing us to identify:

- High bias (underfitting)  
- High variance (overfitting)  
- Whether more data would help 


---

# 8. Advanced Model — Random Forest

We train a non-linear ensemble model capable of capturing complex interactions.

### Included steps
- Fit a Random Forest Regressor  
- Evaluate RMSE and R²  
- Analyze feature importances  

The Random Forest often provides strong predictive performance and helps reveal which features are truly influential.










 





---

# 9. Results and Discussion

In this section, we summarize and interpret the outcomes:

### Performance comparison
- OLS  
- Ridge Regression  
- Random Forest  

### Error patterns
Residual analysis to diagnose where models perform poorly (e.g., peak hours, extreme weather).

### Interpretation
Discuss why certain models performed better and what key factors drive bike rental demand.




---

# 10. Limitations and Future Work

### Limitations
- Dataset may not include all relevant drivers (events, holidays, bike availability).  
- Log-transform introduces mild bias when converting predictions back.  
- Standard regression models do not explicitly model temporal dependencies.  
- Random Forest lacks interpretability compared to linear models.  

### Future Work
- Add lagged features and rolling windows to capture temporal structure.  
- Explore time-series models (Prophet, ARIMA, LSTM).  
- Include richer weather and event-related datasets.  
- Investigate probabilistic forecasting for uncertainty quantification.  

---